# Notebook Crash-0: Data Preprocessing — d3plot to Zarr

**Series:** End-to-End AI for Science — Crash Simulation Track  
**Dataset:** BumperBeam (LS-DYNA / HuggingFace)  
**Curator branch:** `main-backup` (has Zarr output support)

---

## What This Notebook Covers

| Section | What You Will Learn |
|---------|-------------------|
| 1. Installation | How to install physicsnemo-curator from the `main-backup` branch |
| 2. d3plot Format | How LS-DYNA stores crash data, which variables live in d3plot files |
| 3. Dataset Download | Fetching BumperBeam runs from HuggingFace |
| 4. Raw Inspection | Opening a d3plot file with lasso-python, reading ArrayType fields |
| 5. Pipeline Flowchart | Visual overview of the full ETL pipeline |
| 6. ETL Execution | Running the curator to produce Zarr stores |
| 7. Validation | Verifying the Zarr output and inspecting arrays |
| 8. Visualization | Matplotlib deformation plots from Zarr data |

---

## Prerequisites

- Python 3.10+
- CUDA-capable GPU (optional for this notebook — preprocessing is CPU-only)
- ~6 GB disk space for the BumperBeam dataset

## Series Overview

```mermaid
flowchart LR
    SG[<b>Notebook Crash-Sim</b><br/>OpenRadioss SimGen<br/><i>generate d3plot dataset</i>]
    C0[<b>Notebook Crash-0</b><br/>Data Preprocessing<br/><i>d3plot to Zarr ETL</i>]
    C1[<b>Notebook Crash-1</b><br/>Architecture and Concepts<br/><i>GeoTransolver theory</i>]
    C2[<b>Notebook Crash-2</b><br/>Training and Comparison<br/><i>train and evaluate</i>]

    SG -->|"d3plot files"| C0
    C0 -->|"Zarr stores"| C2
    C1 -.->|"concepts"| C2

    style C0 fill:#1a6b3a,color:#fff,stroke:#0d4a28,stroke-width:3px
```

> **You are here:** Notebook Crash-0 — converting raw simulation output into ML-ready Zarr stores.


## Section 1 — Installation

The Zarr output capability lives on the `main-backup` branch of the PhysicsNeMo-Curator repository.  
The `main` branch does **not** have the crash Zarr sink yet — you must use `main-backup`.

### What Gets Installed

| Package | Purpose |
|---------|---------|
| `physicsnemo-curator[mesh]` | ETL framework + mesh utilities |
| `lasso-python` | LS-DYNA binary reader (reads d3plot files) |
| `zarr` | Zarr array storage/reading |
| `huggingface_hub` | Dataset download |
| `hydra-core` | Configuration management for the ETL |
| `numpy`, `matplotlib` | Data handling and visualization |

> **Note on `lasso-python`:** This is NVIDIA's fork of the open-source lasso library,  
> optimised for reading LS-DYNA binary formats without requiring a local LS-DYNA licence.

In [ ]:
# ── Step 1: Install physicsnemo-curator from the main-backup branch ─────────────
# This branch contains the CrashZarrDataSource sink (Zarr output capability).
# The main branch does NOT have Zarr support for crash data yet.

!pip install     "git+https://github.com/NVIDIA/physicsnemo-curator.git@main-backup#egg=physicsnemo-curator[mesh]"     --quiet

# ── Step 2: Install lasso-python (LS-DYNA binary reader) ─────────────────────
# lasso-python parses d3plot binary files without needing an LS-DYNA licence.
!pip install lasso-python --quiet

# ── Step 3: Install remaining dependencies ────────────────────────────────────
!pip install zarr huggingface_hub hydra-core omegaconf numpy matplotlib --quiet

print("All packages installed.")

In [ ]:
# ── Verify installations ──────────────────────────────────────────────────────
import importlib, sys

packages = {
    "physicsnemo_curator": "physicsnemo-curator",
    "lasso":               "lasso-python",
    "zarr":                "zarr",
    "huggingface_hub":     "huggingface_hub",
    "hydra":               "hydra-core",
    "numpy":               "numpy",
    "matplotlib":          "matplotlib",
}

all_ok = True
for mod, pkg in packages.items():
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, "__version__", "?")
        print(f"  OK  {pkg:30s} version={ver}")
    except ImportError:
        print(f"  MISSING  {pkg}")
        all_ok = False

if all_ok:
    print("\nAll dependencies satisfied.")
else:
    print("\nSome packages missing — re-run the install cell above.")

## Section 2 — How LS-DYNA Stores Crash Data in d3plot Files

### 2.1 What Is a d3plot File?

LS-DYNA is the industry-standard finite element solver for crash simulation.  
When a simulation runs, it writes output to a sequence of binary files:

```
Run100/
├── d3plot        ← main file: mesh geometry + first few states
├── d3plot01      ← continuation: states 4–6
├── d3plot02      ← continuation: states 7–9
├── ...
├── d3plot11      ← last file (for 51-timestep BumperBeam runs)
└── run100.k      ← keyword deck: part/section definitions (ASCII)
```

`lasso-python` automatically discovers and reads **all** continuation files —  
you only point it at `d3plot` and it loads the full trajectory.

---

### 2.2 Internal Variable Names (ArrayType Enum)

You do **not** need to know the raw binary variable IDs inside d3plot files.  
`lasso-python` exposes a typed `ArrayType` enum that maps to the correct binary offsets:

| `ArrayType` constant | Shape | Units | Description |
|---------------------|-------|-------|-------------|
| `node_coordinates` | `(N, 3)` | mm | Reference node positions at t=0 (undeformed mesh) |
| `node_displacement` | `(T, N, 3)` | mm | Cumulative displacement at each timestep |
| `element_shell_node_indexes` | `(E, 4)` | — | Shell element connectivity (node indices, 0-based) |
| `element_shell_part_indexes` | `(E,)` | — | Part index for each shell element |
| `part_ids` | `(P,)` | — | Actual LS-DYNA part IDs |
| `element_shell_stress` | `(T, E, 2, 6)` | MPa | Voigt stress tensor per element per layer (optional) |
| `element_shell_effective_plastic_strain` | `(T, E, 2)` | — | EPS per element per integration layer (optional) |

**Key distinction:**  
- `node_coordinates` = the **reference** (t=0) mesh positions — these never change  
- `node_displacement` = displacement **added to** the reference position at each step  
- Absolute position at time t: `coords + pos_raw[t]`  
- The Zarr store saves **absolute positions** (`mesh_pos = coords + disp`), not raw displacements

---

### 2.3 The .k Keyword File (Thickness)

The `.k` file is an ASCII keyword deck. It contains:

```
*PART
$# title
BumperBeam_Panel
$#     pid     secid      mid
         1         1        1
*SECTION_SHELL
$#   secid    elform
         1         2
$#      t1        t2        t3        t4
       1.5       1.5       1.5       1.5
```

- `*PART` links a **part ID** to a **section ID**
- `*SECTION_SHELL` defines the **shell thickness** (t1–t4 in mm)
- `lasso-python` does NOT parse `.k` files — `physicsnemo-curator` handles this parsing
- If no `.k` file is found, all node thicknesses default to 0.0

---

### 2.4 What the Curator Extracts from BumperBeam

For each simulation run, the curator reads and stores:

```
d3plot  →  node_coordinates       (N_raw, 3)        reference positions
        →  node_displacement      (T, N_raw, 3)     per-timestep displacements
        →  element connectivity   (E_raw, 4)         shell element triangles/quads
        →  part IDs               (E_raw,)           which part each element belongs to

.k file →  thickness              (N_filtered,)      per-node shell thickness

curator →  wall node filter       removes ~40-60% of nodes (rigid wall nodes)
        →  edge building          unique undirected edges from connectivity
        →  index remapping        compact 0-based indices

Zarr    →  mesh_pos               (T, N, 3)          absolute node positions
        →  thickness              (N,)               per-node thickness
        →  edges                  (E, 2)             edge connectivity
```

Where `N = N_raw - N_wall` (wall nodes removed) and `T = 51` for BumperBeam.

## Section 2.1 — Do We Need to Provide d3plot Variable Names?

**Short answer: No.**

`lasso-python` uses the `ArrayType` enum to abstract away the binary variable codes.  
You never need to look up LS-DYNA's internal `NTYPE`, `ICODE`, or result-database IDs.

The curator's `CrashD3PlotDataSource` reads exactly these arrays by name:

```python
from lasso.dyna import ArrayType, D3plot

dp = D3plot("path/to/d3plot")

# These are the only variables the curator reads from d3plot:
coords            = dp.arrays[ArrayType.node_coordinates]           # (N, 3)
pos_raw           = dp.arrays[ArrayType.node_displacement]          # (T, N, 3)
mesh_connectivity = dp.arrays[ArrayType.element_shell_node_indexes] # (E, 4)
part_ids          = dp.arrays[ArrayType.element_shell_part_indexes] # (E,)
actual_part_ids   = dp.arrays[ArrayType.part_ids]                   # (P,)

# Optional (if present in the file):
stress            = dp.arrays[ArrayType.element_shell_stress]       # (T, E, 2, 6)
eps               = dp.arrays[ArrayType.element_shell_effective_plastic_strain]  # (T, E, 2)
```

The `ArrayType` enum is defined in `lasso.dyna` — you can inspect all available fields  
by running `[a for a in ArrayType]` in Cell 4 below.

**For the BumperBeam dataset**, the curator uses only:
- `node_coordinates` — reference mesh  
- `node_displacement` — trajectory  
- `element_shell_node_indexes` — connectivity  
- `element_shell_part_indexes` — for thickness mapping  
- `.k` file keywords `*PART` and `*SECTION_SHELL` — for thickness values

In [ ]:
import os
from pathlib import Path

# ── Project paths (adjust to your environment) ────────────────────────────────
DATA_ROOT   = Path("../data")
RAW_DIR     = DATA_ROOT / "bumperbeam_raw"       # downloaded d3plot files
ZARR_DIR    = DATA_ROOT / "bumperbeam_zarr"      # output Zarr stores
CURATOR_DIR = Path("../physicsnemo-curator-crash")  # cloned repo (matches Section 6)

RAW_DIR.mkdir(parents=True, exist_ok=True)
ZARR_DIR.mkdir(parents=True, exist_ok=True)

print("Raw data  :", RAW_DIR)
print("Zarr store:", ZARR_DIR)

## Section 2.2 — Exploring All Available ArrayType Fields

Run this cell to see every variable that `lasso-python` can read from a d3plot file.  
Most BumperBeam d3plot files only contain the mandatory fields (coordinates, displacement,  
connectivity) — optional fields like stress are simulation-configuration-dependent.

In [ ]:
from lasso.dyna import ArrayType

# Print all available ArrayType constants grouped by category
print("All ArrayType fields available in lasso-python:")
print()

categories = {
    "Node":    [a for a in ArrayType if "node" in a.value.lower()],
    "Element": [a for a in ArrayType if "element" in a.value.lower()],
    "Part":    [a for a in ArrayType if "part" in a.value.lower()],
    "Other":   [a for a in ArrayType if "node" not in a.value.lower()
                                     and "element" not in a.value.lower()
                                     and "part" not in a.value.lower()],
}

for cat, fields in categories.items():
    print(f"  {cat} fields:")
    for f in fields:
        print(f"    ArrayType.{f.name:55s} = '{f.value}'")
    print()

## Section 3 — Dataset: HuggingFace or Self-Generated

> **Two ways to get the training data:**
>
> | Option | Source | When to use |
> |--------|--------|-------------|
> | **A — HuggingFace** (this section) | Pre-computed ~55-run dataset | Quickest start, no simulation needed |
> | **B — SimGen** (`Notebook_Crash_SimGen-OpenRadioss`) | Self-generated via OpenRadioss | When you want to customise the DoE or use your own geometry |
>
> If you used **Option B**, skip the download cell and set `RAW_DIR` to point at your `data/bumperbeam_openradioss/RAW_DATA/` folder instead.

### Option A — Download from HuggingFace

The BumperBeam dataset contains ~55 simulation runs.  
Each run folder has d3plot continuation files (d3plot, d3plot01–d3plot11) and a `.k` file.

**Full dataset:** ~6.4 GB  
**Selective download:** We use `allow_patterns` to fetch only the runs we need.

The dataset lives at:  
`https://huggingface.co/datasets/AIRBORNEPANDA/BumperBeamCrashExample`

In [ ]:
from huggingface_hub import snapshot_download
import os

# ── Choose how many runs to download ──────────────────────────────────────────
# For a quick demo: 5 training + 2 validation runs (~700 MB)
# For full training: all 50+ runs (~6 GB)

DEMO_MODE = True   # Set False to download all runs

if DEMO_MODE:
    # Selective: Run100-Run104 (training) + Run150-Run151 (validation)
    train_patterns = [f"RAW_DATA/Run10{i}/*" for i in range(5)]
    val_patterns   = [f"RAW_DATA/Run15{i}/*" for i in range(2)]
    allow_patterns = train_patterns + val_patterns
    print(f"Demo mode: downloading {len(allow_patterns)} run patterns (~700 MB)")
else:
    # Full dataset
    allow_patterns = ["RAW_DATA/*"]
    print("Full mode: downloading all runs (~6.4 GB)")

local_dir = snapshot_download(
    repo_id="AIRBORNEPANDA/BumperBeamCrashExample",
    repo_type="dataset",
    allow_patterns=allow_patterns,
    local_dir=str(RAW_DIR),
)

print(f"\nDownloaded to: {local_dir}")

# ── Show what was downloaded ──────────────────────────────────────────────────
raw_data_dir = RAW_DIR / "RAW_DATA"
if raw_data_dir.exists():
    runs = sorted(raw_data_dir.iterdir())
    print(f"\nRuns downloaded: {len(runs)}")
    for run in runs[:5]:
        files = list(run.iterdir())
        print(f"  {run.name}: {len(files)} files ({', '.join(f.name for f in files[:4])}...)")
    if len(runs) > 5:
        print(f"  ... and {len(runs)-5} more")

## Section 4 — Inspecting a Raw d3plot File with lasso-python

Before running the ETL pipeline, it is instructive to open one d3plot file directly  
and inspect the raw arrays. This builds intuition for what the curator is reading.

In [ ]:
from lasso.dyna import ArrayType, D3plot
import numpy as np

# ── Find first run ────────────────────────────────────────────────────────────
raw_data_dir = RAW_DIR / "RAW_DATA"
run_dirs = sorted(raw_data_dir.iterdir()) if raw_data_dir.exists() else []

if not run_dirs:
    print("No runs found — run the download cell first.")
else:
    first_run = run_dirs[0]
    d3plot_path = first_run / "d3plot"
    print(f"Opening: {d3plot_path}")
    print()

    # ── Load d3plot (lasso auto-discovers d3plot01, d3plot02 etc.) ─────────────
    dp = D3plot(str(d3plot_path))

    # ── Report which arrays are present ────────────────────────────────────────
    print("Arrays present in this d3plot file:")
    print(f"  {'ArrayType constant':55s}  {'shape':25s}  dtype")
    print("  " + "-" * 90)
    for arr_type, arr in dp.arrays.items():
        if hasattr(arr, 'shape'):
            print(f"  ArrayType.{arr_type.name:50s}  {str(arr.shape):25s}  {arr.dtype}")

    print()

    # ── Extract the key arrays used by the curator ─────────────────────────────
    coords  = dp.arrays[ArrayType.node_coordinates]          # (N, 3)
    pos_raw = dp.arrays[ArrayType.node_displacement]         # (T, N, 3)
    conn    = dp.arrays[ArrayType.element_shell_node_indexes] # (E, 4)
    p_ids   = dp.arrays[ArrayType.element_shell_part_indexes] # (E,)

    T, N_raw, _ = pos_raw.shape
    E_raw       = conn.shape[0]

    print(f"Simulation summary for {first_run.name}:")
    print(f"  Timesteps (T)          : {T}")
    print(f"  Nodes (N_raw)          : {N_raw:,}")
    print(f"  Shell elements (E_raw) : {E_raw:,}")
    print(f"  Simulation duration    : {(T-1)*5} ms  (dt = 5 ms per step)")
    print()
    print(f"  Reference coords range : x=[{coords[:,0].min():.1f}, {coords[:,0].max():.1f}] mm")
    print(f"                           y=[{coords[:,1].min():.1f}, {coords[:,1].max():.1f}] mm")
    print(f"                           z=[{coords[:,2].min():.1f}, {coords[:,2].max():.1f}] mm")
    print()
    print(f"  Max displacement (any node, any time): {np.abs(pos_raw).max():.2f} mm")
    print(f"  Unique part IDs: {np.unique(p_ids).tolist()}")

In [ ]:
# ── Visualize initial mesh geometry (scatter, colored by z) ──────────────────
# NOTE: Using matplotlib 3D scatter — PyVista/VTK is NOT used because
#       it causes SIGSEGV (OpenGL crash) in the physicsnemo:26.05 container on H100.

import matplotlib.pyplot as plt

if run_dirs:
    fig = plt.figure(figsize=(14, 5))

    for col, (t_idx, title) in enumerate([(0, "t=0 ms (reference)"),
                                           (T//2, f"t={T//2*5} ms (mid-crash)"),
                                           (T-1, f"t={(T-1)*5} ms (final)")]):
        ax = fig.add_subplot(1, 3, col+1, projection='3d')

        # Absolute position = reference + displacement
        pos_t = coords + pos_raw[t_idx]

        # Subsample for speed
        idx = np.random.choice(N_raw, min(2000, N_raw), replace=False)
        p = pos_t[idx]

        sc = ax.scatter(p[:,0], p[:,1], p[:,2],
                        c=p[:,2], cmap='coolwarm', s=1, alpha=0.7)
        ax.set_xlabel('X (mm)', fontsize=7)
        ax.set_ylabel('Y (mm)', fontsize=7)
        ax.set_zlabel('Z (mm)', fontsize=7)
        ax.set_title(title, fontsize=9)
        ax.tick_params(labelsize=6)

    plt.suptitle(f"BumperBeam raw d3plot — {first_run.name}", fontsize=11)
    plt.tight_layout()
    plt.savefig("raw_d3plot_geometry.png", dpi=130, bbox_inches='tight')
    plt.show()
    print("Saved: raw_d3plot_geometry.png")

## Section 5 — Full ETL Pipeline Flowchart

The diagram below shows every stage of the curator pipeline,  
from raw d3plot files on disk to the Zarr stores used for ML training.

```mermaid
flowchart TD
    subgraph SRC ["Data Source — pick one"]
        HF[("<b>Option A · HuggingFace</b><br/>bumperbeam-openradioss<br/>135 pre-computed runs")]
        SG[("<b>Option B · Notebook Crash-Sim</b><br/>your own OpenRadioss sweep<br/>custom DoE / geometry")]
    end

    HF --> D3
    SG --> D3

    D3[("<b>Raw d3plot layout</b><br/>RAW_DATA/Run0001/d3plot, d3plot01, ...<br/>+ run0001.k (shell thickness)<br/>+ global_features.json")]

    D3 --> L

    L[<b>lasso-python D3plot reader</b><br/>node_coordinates [N,3]<br/>node_displacement [T,N,3]<br/>element_shell_stress, element_shell_effective_plastic_strain]

    L -->|"raw arrays per run"| ETL

    subgraph ETLBOX ["physicsnemo-curator ETL"]
        ETL[<b>openradioss_to_zarr.py</b>]
        F1[Filter: drop rigid-wall nodes<br/>wall_threshold]
        F2[Normalize: per-field mean / std]
        F3[Split: train / val / test]
        ETL --> F1 --> F2 --> F3
    end

    F3 --> Z

    Z[("<b>Zarr store</b><br/>data/bumperbeam_zarr/<br/>train/ val/ test/<br/>chunked, compressed arrays")]

    Z --> V1
    Z --> V2
    Z --> V3
    Z --> NB2

    V1[<b>Section 7 · Validate</b><br/>shape + dtype checks]
    V2[<b>Section 7.1 · Statistics</b><br/>displacement histograms]
    V3[<b>Section 8 · Visualize</b><br/>deformation animation]
    NB2[<b>Notebook Crash-2</b><br/>GeoTransolver training]

    style HF fill:#e07b00,color:#fff
    style SG fill:#e07b00,color:#fff
    style L fill:#4a7fb5,color:#fff
    style ETL fill:#6a5acd,color:#fff
    style Z fill:#1a6b3a,color:#fff
    style NB2 fill:#1a6b3a,color:#fff
```

**What the curator ETL actually does:**

| Stage | Operation | Why |
|-------|-----------|-----|
| **Read** | `lasso.dyna.D3plot` opens the binary | d3plot is a packed FORTRAN binary — needs a proper parser |
| **Filter** | Remove nodes belonging to the rigid wall | The wall is not part of the deformable structure — including it corrupts the target field |
| **Normalize** | Subtract mean, divide by std per field | Displacement (mm) and stress (MPa) differ by orders of magnitude |
| **Split** | Partition runs into train / val / test | Prevents leakage; splits are at the *run* level, not the timestep level |
| **Write** | Chunked, compressed Zarr arrays | Random access per-run without loading 117 MB into RAM |


## Section 6 — Set Up the Curator ETL Script

We clone **only** the crash example directory from the `main-backup` branch  
(sparse checkout — avoids downloading the full repository).

In [ ]:
import subprocess
from pathlib import Path

CURATOR_CRASH_DIR = Path("../physicsnemo-curator-crash")

if not CURATOR_CRASH_DIR.exists():
    print("Cloning physicsnemo-curator main-backup (sparse — crash example only)...")
    steps = [
        f"git clone --filter=blob:none --no-checkout "
        f"https://github.com/NVIDIA/physicsnemo-curator.git {CURATOR_CRASH_DIR}",
        f"cd {CURATOR_CRASH_DIR} && git sparse-checkout init --cone",
        f"cd {CURATOR_CRASH_DIR} && git sparse-checkout set examples/structural_mechanics/crash",
        f"cd {CURATOR_CRASH_DIR} && git checkout main-backup",
    ]
    for cmd in steps:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        status = "OK" if r.returncode == 0 else f"FAILED: {r.stderr[:200]}"
        print(f"  {status}  {cmd[:70]}")
else:
    print("Curator crash directory already exists.")

CRASH_ETL_DIR = CURATOR_CRASH_DIR / "examples" / "structural_mechanics" / "crash"
print(f"\nETL directory: {CRASH_ETL_DIR}")
print("Files present:")
for f in sorted(CRASH_ETL_DIR.rglob("*.py")) + sorted(CRASH_ETL_DIR.rglob("*.yaml")):
    print(f"  {f.relative_to(CRASH_ETL_DIR)}")

## Section 6.1 — Run the ETL Pipeline (d3plot → Zarr)

The `run_etl.py` script uses [Hydra](https://hydra.cc/) for configuration.  
Key parameters:

| Parameter | Default | Description |
|-----------|---------|-------------|
| `etl.source.input_dir` | required | Directory containing `Run*/d3plot` folders |
| `serialization_format` | `vtp` | Set to `zarr` for ML training output |
| `serialization_format.sink.output_dir` | required | Where to write Zarr stores |
| `serialization_format.sink.compression_level` | 3 | 1=fast, 9=smallest |
| `serialization_format.sink.chunk_size_mb` | 1.0 | I/O chunk size |
| `etl.transformations.crash_transform.wall_threshold` | 1.0 | mm; higher = more filtering |
| `etl.processing.num_processes` | 12 | Parallel runs to process |

In [ ]:
import subprocess, sys
from pathlib import Path

CRASH_ETL_DIR = Path("../physicsnemo-curator-crash/examples/structural_mechanics/crash")
INPUT_DIR     = Path("../data/bumperbeam_raw/RAW_DATA")
OUTPUT_DIR    = Path("../data/bumperbeam_zarr")

# Verify inputs exist before launching
if not INPUT_DIR.exists() or not any(INPUT_DIR.iterdir()):
    print(f"Input directory empty or missing: {INPUT_DIR}")
    print("Run the HuggingFace download cell first.")
else:
    print(f"Input runs found: {len(list(INPUT_DIR.iterdir()))}")
    print(f"Output will go to: {OUTPUT_DIR}")
    print()

    cmd = [
        sys.executable, "run_etl.py",
        "--config-dir=config",
        "--config-name=crash_etl",
        f"etl.source.input_dir={INPUT_DIR.resolve()}",
        "serialization_format=zarr",
        f"serialization_format.sink.output_dir={OUTPUT_DIR.resolve()}",
        "serialization_format.sink.compression_level=3",
        "serialization_format.sink.chunk_size_mb=2.0",
        "etl.transformations.crash_transform.wall_threshold=1.0",
        "etl.processing.num_processes=4",
    ]

    print("Command:")
    print(" ".join(cmd))
    print()
    print("Running ETL pipeline...")

    result = subprocess.run(
        cmd,
        cwd=str(CRASH_ETL_DIR.resolve()),
        capture_output=True,
        text=True,
        timeout=3600,
    )

    if result.returncode == 0:
        print("ETL pipeline completed successfully!")
    else:
        print("ETL FAILED. stderr:")
        print(result.stderr[-2000:])

    print("\nSTDOUT (last 30 lines):")
    lines = result.stdout.strip().split('\n')
    print('\n'.join(lines[-30:]))

### Alternative: Run from Terminal

You can also run the ETL from a terminal, which gives you live progress output:

```bash
cd ../physicsnemo-curator-crash/examples/structural_mechanics/crash

python run_etl.py \
    --config-dir=config \
    --config-name=crash_etl \
    etl.source.input_dir=../../../data/bumperbeam_raw/RAW_DATA \
    serialization_format=zarr \
    serialization_format.sink.output_dir=../../../data/bumperbeam_zarr \
    serialization_format.sink.compression_level=3 \
    serialization_format.sink.chunk_size_mb=2.0 \
    etl.transformations.crash_transform.wall_threshold=1.0 \
    etl.processing.num_processes=4
```

**Typical runtime:** ~2–5 minutes per run on a modern CPU (depends on node count).  
With 4 parallel processes and 50 runs, expect ~25–60 minutes total.

## Section 6.2 — Organise Zarr Stores into Train / Val Splits

The ETL writes one flat store per run:

```
data/bumperbeam_zarr/
├── Run100.zarr/
├── Run101.zarr/
└── ...
```

`Notebook_Crash2` expects them grouped into splits, with `run_` prefixed names:

```
data/bumperbeam_zarr/
├── train/
│   ├── run_100/
│   └── run_101/
└── val/
    └── run_150/
```

> **This step is required.** Without it, Crash-2 Section 3 reports
> `Training runs: 0` and training cannot start.

The cell below moves the stores into place. Splits are made at the **run** level —
never split by timestep, which would leak information from a run into both sets.


In [ ]:
import shutil, re
from pathlib import Path

ZARR_DIR  = Path("../data/bumperbeam_zarr")
VAL_FRAC  = 0.2      # fraction of runs held out for validation
SEED      = 42

train_dir = ZARR_DIR / "train"
val_dir   = ZARR_DIR / "val"

# Flat stores written by the ETL (ignore the split dirs themselves)
flat_stores = sorted(
    p for p in ZARR_DIR.glob("*.zarr")
    if p.is_dir() and p.parent == ZARR_DIR
)

already_split = list(train_dir.glob("run_*")) if train_dir.exists() else []

if already_split and not flat_stores:
    print(f"Already organised: {len(already_split)} train / "
          f"{len(list(val_dir.glob('run_*')))} val")
elif not flat_stores:
    print(f"No .zarr stores found in {ZARR_DIR}")
    print("Run the ETL cell (Section 6.1) first.")
else:
    train_dir.mkdir(parents=True, exist_ok=True)
    val_dir.mkdir(parents=True, exist_ok=True)

    # Deterministic split so re-running gives the same partition
    import random
    rng = random.Random(SEED)
    stores = sorted(flat_stores)
    rng.shuffle(stores)

    n_val   = max(1, int(len(stores) * VAL_FRAC))
    val_set = set(stores[:n_val])

    print(f"Splitting {len(stores)} runs → "
          f"{len(stores)-n_val} train / {n_val} val\n")

    for src in sorted(flat_stores):
        # Run100.zarr → run_100
        m = re.search(r'(\d+)', src.stem)
        run_id = m.group(1) if m else src.stem
        dest_name = f"run_{run_id}"

        dest_parent = val_dir if src in val_set else train_dir
        dest = dest_parent / dest_name

        if dest.exists():
            print(f"  skip (exists): {dest.relative_to(ZARR_DIR)}")
            continue

        shutil.move(str(src), str(dest))
        print(f"  {src.name}  →  {dest.relative_to(ZARR_DIR)}")

    print()

# ── Verify the layout Crash-2 expects ─────────────────────────────────────────
print("Final layout:")
for split in ("train", "val"):
    d = ZARR_DIR / split
    runs = sorted(d.glob("run_*")) if d.exists() else []
    print(f"  {split}/  {len(runs)} run(s)  {[r.name for r in runs[:4]]}"
          f"{' ...' if len(runs) > 4 else ''}")

if not (ZARR_DIR / "train").exists() or not list((ZARR_DIR / "train").glob("run_*")):
    print("\nWARNING: no train/run_* found — Crash-2 will report 0 training runs.")
else:
    print("\nReady for Notebook_Crash2.")

## Section 7 — Validate the Zarr Output

After the ETL finishes, inspect the Zarr stores to verify correct shapes and data ranges.

In [ ]:
import zarr
import numpy as np
import json
from pathlib import Path

ZARR_DIR = Path("../data/bumperbeam_zarr")

zarr_stores = sorted(ZARR_DIR.glob("*.zarr"))
print(f"Zarr stores found: {len(zarr_stores)}")
print()

if zarr_stores:
    # ── Inspect first store ────────────────────────────────────────────────────
    store_path = zarr_stores[0]
    store = zarr.open(str(store_path), mode='r')

    print(f"Store: {store_path.name}")
    print(f"  Attributes: {dict(store.attrs)}")
    print()
    print("  Arrays:")
    for key in store.array_keys():
        arr = store[key]
        print(f"    {key:15s}  shape={str(arr.shape):20s}  dtype={arr.dtype}  "
              f"chunks={arr.chunks}  compression={arr.compressor}")
    print()

    # ── Load and verify mesh_pos ───────────────────────────────────────────────
    mesh_pos  = store['mesh_pos'][:]    # (T, N, 3)
    thickness = store['thickness'][:]   # (N,)
    edges     = store['edges'][:]       # (E, 2)

    T, N, _ = mesh_pos.shape
    E = edges.shape[0]

    print(f"  mesh_pos   : shape={mesh_pos.shape}  dtype={mesh_pos.dtype}")
    print(f"               x=[{mesh_pos[:,:,0].min():.1f}, {mesh_pos[:,:,0].max():.1f}] mm")
    print(f"               y=[{mesh_pos[:,:,1].min():.1f}, {mesh_pos[:,:,1].max():.1f}] mm")
    print(f"               z=[{mesh_pos[:,:,2].min():.1f}, {mesh_pos[:,:,2].max():.1f}] mm")
    print(f"  thickness  : shape={thickness.shape}  min={thickness.min():.2f}  "
          f"max={thickness.max():.2f}  mean={thickness.mean():.2f} mm")
    print(f"  edges      : shape={edges.shape}  max_idx={edges.max()}  "
          f"(should be < N={N})")
    print()

    # ── Summary across all stores ──────────────────────────────────────────────
    print("Summary across all Zarr stores:")
    print(f"  {'Store':15s}  {'T':>4s}  {'N':>7s}  {'E':>8s}  thick_mean")
    for sp in zarr_stores:
        s = zarr.open(str(sp), mode='r')
        mp = s['mesh_pos']
        th = s['thickness'][:]
        ed = s['edges']
        print(f"  {sp.stem:15s}  {mp.shape[0]:>4d}  {mp.shape[1]:>7,d}  "
              f"{ed.shape[0]:>8,d}  {th.mean():.2f} mm")

## Section 7.1 — Dataset Statistics

Understanding the distribution of node counts and deformation magnitudes  
helps set reasonable model hyperparameters (e.g., number of Physics-Attention slices).

In [ ]:
import zarr, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

ZARR_DIR = Path("../data/bumperbeam_zarr")
zarr_stores = sorted(ZARR_DIR.glob("*.zarr"))

if zarr_stores:
    stats = []
    for sp in zarr_stores:
        s = zarr.open(str(sp), mode='r')
        mp = s['mesh_pos'][:]
        th = s['thickness'][:]
        # Max displacement = max distance from t=0 position
        disp = np.linalg.norm(mp - mp[0:1], axis=-1)   # (T, N)
        stats.append({
            'name': sp.stem,
            'T': mp.shape[0], 'N': mp.shape[1], 'E': s['edges'].shape[0],
            'max_disp_mm': disp.max(),
            'mean_thick_mm': th.mean(),
        })

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    node_counts = [s['N'] for s in stats]
    max_disps   = [s['max_disp_mm'] for s in stats]
    thicknesses = [s['mean_thick_mm'] for s in stats]

    axes[0].hist(node_counts, bins=10, color='#2196F3', edgecolor='white')
    axes[0].set_xlabel('Number of nodes (N) after filtering')
    axes[0].set_ylabel('Runs')
    axes[0].set_title('Node Count Distribution')
    axes[0].axvline(np.mean(node_counts), color='red', linestyle='--',
                    label=f'mean={np.mean(node_counts):.0f}')
    axes[0].legend(fontsize=8)

    axes[1].hist(max_disps, bins=10, color='#4CAF50', edgecolor='white')
    axes[1].set_xlabel('Max displacement (mm)')
    axes[1].set_ylabel('Runs')
    axes[1].set_title('Peak Deformation Distribution')
    axes[1].axvline(np.mean(max_disps), color='red', linestyle='--',
                    label=f'mean={np.mean(max_disps):.1f} mm')
    axes[1].legend(fontsize=8)

    axes[2].hist(thicknesses, bins=10, color='#FF9800', edgecolor='white')
    axes[2].set_xlabel('Mean node thickness (mm)')
    axes[2].set_ylabel('Runs')
    axes[2].set_title('Shell Thickness Distribution')
    axes[2].axvline(np.mean(thicknesses), color='red', linestyle='--',
                    label=f'mean={np.mean(thicknesses):.2f} mm')
    axes[2].legend(fontsize=8)

    plt.suptitle('BumperBeam Dataset Statistics (post-ETL Zarr)', fontsize=12)
    plt.tight_layout()
    plt.savefig('zarr_dataset_stats.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('Saved: zarr_dataset_stats.png')

## Section 8 — Visualize Deformation from Zarr Data

This confirms the Zarr store contains correct absolute positions  
and that the deformation trajectory looks physically reasonable.

In [ ]:
import zarr, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

ZARR_DIR = Path("../data/bumperbeam_zarr")
zarr_stores = sorted(ZARR_DIR.glob("*.zarr"))

if zarr_stores:
    store = zarr.open(str(zarr_stores[0]), mode='r')
    mesh_pos = store['mesh_pos'][:]   # (T, N, 3)
    T, N, _ = mesh_pos.shape

    VIZ_FRAMES = [0, T//4, T//2, 3*T//4, T-1]
    labels = [f't={i*5} ms' for i in VIZ_FRAMES]

    fig = plt.figure(figsize=(16, 3.5))
    for col, (t_idx, label) in enumerate(zip(VIZ_FRAMES, labels)):
        ax = fig.add_subplot(1, len(VIZ_FRAMES), col+1, projection='3d')
        # Colour nodes by total displacement from t=0
        disp_mag = np.linalg.norm(mesh_pos[t_idx] - mesh_pos[0], axis=-1)
        idx = np.random.choice(N, min(1500, N), replace=False)
        p = mesh_pos[t_idx][idx]
        c = disp_mag[idx]
        ax.scatter(p[:,0], p[:,1], p[:,2], c=c, cmap='plasma', s=1.5,
                   vmin=0, vmax=disp_mag.max())
        ax.set_title(label, fontsize=8)
        ax.set_xlabel('X', fontsize=6)
        ax.set_ylabel('Y', fontsize=6)
        ax.set_zlabel('Z', fontsize=6)
        ax.tick_params(labelsize=5)

    plt.suptitle(f'BumperBeam deformation — {zarr_stores[0].stem}  '
                 f'(N={N:,} nodes, T={T} steps)',
                 fontsize=10)
    plt.tight_layout()
    plt.savefig('zarr_deformation.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('Saved: zarr_deformation.png')

## Section 8.1 — Generate global_features.json

The GeoTransolver training pipeline needs a JSON file with per-run global conditioning variables:  
**velocity_x**, **thickness_scale**, and **rwall_origin_y**.

These are simulation parameters that distinguish one run from another  
(they are not stored inside d3plot files — they come from the simulation setup).

For the BumperBeam HuggingFace dataset, these values are encoded in the run name  
or in a separate metadata file. Here we generate reasonable placeholder values  
(replace with actual values from the simulation setup if available).

In [ ]:
import json, zarr, numpy as np
from pathlib import Path

ZARR_DIR = Path("../data/bumperbeam_zarr")
zarr_stores = sorted(ZARR_DIR.glob("*.zarr"))

global_features = {}
rng = np.random.default_rng(42)

for sp in zarr_stores:
    run_name = sp.stem.replace('.zarr', '')
    s = zarr.open(str(sp), mode='r')
    th = s['thickness'][:]
    mean_thick = float(th.mean()) if th.mean() > 0 else 1.5

    # BumperBeam parameter ranges (from published benchmark):
    #   velocity_x:     8.0 – 14.0 m/s
    #   thickness_scale: 0.8 – 1.2  (relative to nominal 1.5 mm)
    #   rwall_origin_y: 0.0 – 0.5   (wall position offset in m)
    # In practice these come from the .k file or simulation log.
    # We sample plausible values here for demonstration.
    global_features[run_name] = {
        "velocity_x":      round(float(rng.uniform(8.0, 14.0)), 3),
        "thickness_scale": round(mean_thick / 1.5, 4),   # relative to nominal
        "rwall_origin_y":  round(float(rng.uniform(0.0, 0.5)), 4),
    }

out_path = ZARR_DIR / "global_features.json"
with open(out_path, 'w') as f:
    json.dump(global_features, f, indent=2)

print(f"Written: {out_path}  ({len(global_features)} entries)")
print("\nSample entries:")
for run, feats in list(global_features.items())[:3]:
    print(f"  {run}: {feats}")
print()
print("NOTE: Replace these placeholder values with actual simulation parameters")
print("      from the .k file or simulation log before training.")

## Summary

This notebook walked through the complete data preprocessing pipeline for crash simulation:

### What We Covered

1. **Installed** `physicsnemo-curator` from the `main-backup` branch (Zarr capability)
2. **Explained** the d3plot binary format and its `ArrayType` variable names
3. **Answered** the variable-names question: `lasso-python` handles all binary parsing via the `ArrayType` enum — you do not need to know internal LS-DYNA result database codes
4. **Downloaded** the BumperBeam dataset from HuggingFace using selective `allow_patterns`
5. **Inspected** a raw d3plot file to see actual array shapes and data ranges
6. **Visualized** the full ETL pipeline as a flowchart
7. **Ran** the curator ETL to produce Zarr stores
8. **Validated** the Zarr output: shapes, dtypes, compression, chunking
9. **Generated** `global_features.json` for per-run conditioning

### Zarr Store Layout (per run)

```
RunXXX.zarr/
├── mesh_pos    (T=51, N, 3)  float32   absolute node positions, Blosc/zstd
├── thickness   (N,)          float32   per-node shell thickness (mm)
├── edges       (E, 2)        int64     undirected edge pairs
└── .zattrs     {filename, num_timesteps, num_nodes, num_edges, thickness_stats}
```

### Do You Need d3plot Variable Names?

**No.** The `lasso-python` `ArrayType` enum abstracts all binary offsets.  
The curator reads exactly: `node_coordinates`, `node_displacement`,  
`element_shell_node_indexes`, `element_shell_part_indexes` — all via enum constants.

---

**Next:** `Notebook_Crash1-Architecture-and-Concepts.ipynb` — GeoTransolver theory and rollout strategies.